In [1]:
# If you work with Google Colaboratory, please run this as well.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Specify the directory where this notebook is located after %cd.
%cd "/content/drive/MyDrive/GCI_Global/Competition"

/content/drive/MyDrive/GCI_Global/Competition


In [3]:
import os
import random
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
warnings.filterwarnings("ignore")
SEED=42
N_SPLITS=5
np.random.seed(SEED)
random.seed(SEED)

In [4]:
current_dir=Path.cwd()
input_dir=current_dir/"input"
if input_dir.exists():
    train_file=input_dir/"train.csv"
    test_file=input_dir/"test.csv"
    sample_sub_file=input_dir/"sample_submission.csv"
else:
    train_file=current_dir/"train.csv"
    test_file=current_dir/"test.csv"
    sample_sub_file=current_dir/"sample_submission.csv"
assert train_file.exists() and test_file.exists() and sample_sub_file.exists()

In [5]:
train_raw=pd.read_csv(train_file)
test_raw=pd.read_csv(test_file)
sample_sub=pd.read_csv(sample_sub_file)
assert "TARGET" in train_raw.columns and "TARGET" not in test_raw.columns
assert list(sample_sub.columns)==["SK_ID_CURR","TARGET"]
assert sample_sub["SK_ID_CURR"].equals(test_raw["SK_ID_CURR"])
y=train_raw["TARGET"].astype(int)
X_raw=train_raw.drop(columns="TARGET")
X_test_raw=test_raw.copy()

In [6]:
def safe_divide(a,b):
    return a/b.replace(0,np.nan)

def add_features(df):
    df=df.copy()
    if "DAYS_EMPLOYED" in df.columns:
        df["DAYS_EMPLOYED_ANOM"]=(df["DAYS_EMPLOYED"]==365243).astype("int8")
        df.loc[df["DAYS_EMPLOYED"]==365243,"DAYS_EMPLOYED"]=np.nan
    if "DAYS_BIRTH" in df.columns: df["AGE_YEARS"]=-df["DAYS_BIRTH"]/365.25
    if "DAYS_EMPLOYED" in df.columns: df["EMPLOYED_YEARS"]=-df["DAYS_EMPLOYED"]/365.25
    if {"DAYS_BIRTH","DAYS_EMPLOYED"}.issubset(df.columns): df["EMPLOYED_AGE_RATIO"]=safe_divide(-df["DAYS_EMPLOYED"],-df["DAYS_BIRTH"])
    if {"DAYS_REGISTRATION","DAYS_BIRTH"}.issubset(df.columns): df["REGISTRATION_AGE_RATIO"]=safe_divide(-df["DAYS_REGISTRATION"],-df["DAYS_BIRTH"])
    if "DAYS_REGISTRATION" in df.columns: df["REGISTRATION_YEARS"]=-df["DAYS_REGISTRATION"]/365.25
    if "DAYS_ID_PUBLISH" in df.columns: df["ID_DOCUMENT_YEARS"]=-df["DAYS_ID_PUBLISH"]/365.25
    if "DAYS_LAST_PHONE_CHANGE" in df.columns: df["PHONE_CHANGE_YEARS"]=-df["DAYS_LAST_PHONE_CHANGE"]/365.25
    if {"AMT_CREDIT","AMT_INCOME_TOTAL"}.issubset(df.columns):
        df["CREDIT_INCOME_RATIO"]=safe_divide(df["AMT_CREDIT"],df["AMT_INCOME_TOTAL"])
        df["INCOME_CREDIT_RATIO"]=safe_divide(df["AMT_INCOME_TOTAL"],df["AMT_CREDIT"])
    if {"AMT_ANNUITY","AMT_INCOME_TOTAL"}.issubset(df.columns):
        df["ANNUITY_INCOME_RATIO"]=safe_divide(df["AMT_ANNUITY"],df["AMT_INCOME_TOTAL"])
        df["INCOME_ANNUITY_RATIO"]=safe_divide(df["AMT_INCOME_TOTAL"],df["AMT_ANNUITY"])
    if {"AMT_CREDIT","AMT_ANNUITY"}.issubset(df.columns): df["CREDIT_ANNUITY_RATIO"]=safe_divide(df["AMT_CREDIT"],df["AMT_ANNUITY"])
    if {"AMT_GOODS_PRICE","AMT_CREDIT"}.issubset(df.columns):
        df["GOODS_CREDIT_RATIO"]=safe_divide(df["AMT_GOODS_PRICE"],df["AMT_CREDIT"])
        df["CREDIT_GOODS_GAP"]=df["AMT_CREDIT"]-df["AMT_GOODS_PRICE"]
    if {"AMT_INCOME_TOTAL","CNT_FAM_MEMBERS"}.issubset(df.columns): df["INCOME_PER_FAMILY"]=safe_divide(df["AMT_INCOME_TOTAL"],df["CNT_FAM_MEMBERS"].clip(lower=1))
    if {"AMT_CREDIT","CNT_FAM_MEMBERS"}.issubset(df.columns): df["CREDIT_PER_FAMILY"]=safe_divide(df["AMT_CREDIT"],df["CNT_FAM_MEMBERS"].clip(lower=1))
    ext=[c for c in ["EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3"] if c in df.columns]
    if ext:
        df["EXT_SOURCE_MEAN"]=df[ext].mean(axis=1)
        df["EXT_SOURCE_MEDIAN"]=df[ext].median(axis=1)
        df["EXT_SOURCE_MIN"]=df[ext].min(axis=1)
        df["EXT_SOURCE_MAX"]=df[ext].max(axis=1)
        df["EXT_SOURCE_RANGE"]=df[ext].max(axis=1)-df[ext].min(axis=1)
        df["EXT_SOURCE_STD"]=df[ext].std(axis=1)
        df["EXT_SOURCE_MISSING"]=df[ext].isna().sum(axis=1)
        for c in ext:
            df[c+"_SQ"]=df[c]**2
            df[c+"_CUBE"]=df[c]**3
        for i in range(len(ext)):
            for j in range(i+1,len(ext)):
                a,b=ext[i],ext[j]
                df[a+"_"+b+"_PROD"]=df[a]*df[b]
                df[a+"_"+b+"_DIFF"]=df[a]-df[b]
                df[a+"_"+b+"_ABS_DIFF"]=(df[a]-df[b]).abs()
                df[a+"_"+b+"_RATIO"]=safe_divide(df[a],df[b])
    bureau=[c for c in df.columns if c.startswith("AMT_REQ_CREDIT_BUREAU_")]
    if bureau:
        df["BUREAU_REQUEST_TOTAL"]=df[bureau].sum(axis=1,min_count=1)
        df["BUREAU_REQUEST_MEAN"]=df[bureau].mean(axis=1)
        df["BUREAU_REQUEST_MAX"]=df[bureau].max(axis=1)
        df["BUREAU_REQUEST_MISSING"]=df[bureau].isna().sum(axis=1)
    if {"CNT_CHILDREN","CNT_FAM_MEMBERS"}.issubset(df.columns):
        df["CHILDREN_FAMILY_RATIO"]=safe_divide(df["CNT_CHILDREN"],df["CNT_FAM_MEMBERS"].clip(lower=1))
        df["ADULTS_IN_FAMILY"]=df["CNT_FAM_MEMBERS"]-df["CNT_CHILDREN"]
    contact=[c for c in ["FLAG_MOBIL","FLAG_EMP_PHONE","FLAG_WORK_PHONE","FLAG_CONT_MOBILE","FLAG_PHONE","FLAG_EMAIL"] if c in df.columns]
    if contact: df["CONTACT_FLAG_SUM"]=df[contact].sum(axis=1)
    df["TOTAL_MISSING"]=df.isna().sum(axis=1)
    df["MISSING_RATIO"]=df.isna().mean(axis=1)
    df=df.replace([np.inf,-np.inf],np.nan)
    return df

X=add_features(X_raw)
X_test=add_features(X_test_raw).reindex(columns=X.columns,fill_value=np.nan)
numeric_cols=X.select_dtypes(include=np.number).columns.tolist()
categorical_cols=X.select_dtypes(include="object").columns.tolist()

In [7]:
def frequency_map(train_df,other_df,cols):
    out=other_df.copy()
    for c in cols:
        m=train_df[c].value_counts(dropna=False,normalize=True)
        out[c+"__FREQ"]=other_df[c].map(m).fillna(0.0)
    return out

def target_map(train_df,y_train,other_df,cols,smoothing=60):
    out=other_df.copy()
    gm=float(np.mean(y_train))
    for c in cols:
        tmp=pd.DataFrame({"x":train_df[c].astype(object).values,"y":np.asarray(y_train)})
        st=tmp.groupby("x",dropna=False)["y"].agg(["mean","count"])
        sm=(st["mean"]*st["count"]+gm*smoothing)/(st["count"]+smoothing)
        out[c+"__TE"]=other_df[c].map(sm).fillna(gm).astype(float)
    return out

cat_te_cols=[c for c in categorical_cols if X[c].nunique(dropna=False)<=80]
try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE=True
except Exception:
    LIGHTGBM_AVAILABLE=False
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE=True
except Exception:
    XGBOOST_AVAILABLE=False

In [8]:
models={}
if LIGHTGBM_AVAILABLE:
    models["lightgbm"]=lambda:LGBMClassifier(n_estimators=1800,learning_rate=0.018,num_leaves=31,min_child_samples=80,subsample=0.85,colsample_bytree=0.80,reg_alpha=0.15,reg_lambda=1.5,objective="binary",metric="auc",random_state=SEED,n_jobs=-1,verbosity=-1)
if XGBOOST_AVAILABLE:
    models["xgboost"]=lambda:XGBClassifier(n_estimators=1400,max_depth=5,learning_rate=0.025,min_child_weight=8,subsample=0.85,colsample_bytree=0.80,reg_alpha=0.10,reg_lambda=2.0,objective="binary:logistic",eval_metric="auc",random_state=SEED,n_jobs=-1,tree_method="hist")
models["hgb"]=lambda:HistGradientBoostingClassifier(learning_rate=0.035,max_iter=700,max_leaf_nodes=31,min_samples_leaf=80,l2_regularization=2.0,random_state=SEED)

In [9]:
skf=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
oof={k:np.zeros(len(X)) for k in models}
test_pred={k:np.zeros(len(X_test)) for k in models}
fold_scores={k:[] for k in models}
for fold,(tr_idx,va_idx) in enumerate(skf.split(X,y),1):
    X_tr,X_va=X.iloc[tr_idx].copy(),X.iloc[va_idx].copy()
    y_tr,y_va=y.iloc[tr_idx],y.iloc[va_idx]
    X_te=X_test.copy()
    tr_num=X_tr.drop(columns=categorical_cols).copy()
    va_num=X_va.drop(columns=categorical_cols).copy()
    te_num=X_te.drop(columns=categorical_cols).copy()
    if categorical_cols:
        tr_e=frequency_map(X_tr,X_tr,categorical_cols)
        va_e=frequency_map(X_tr,X_va,categorical_cols)
        te_e=frequency_map(X_tr,X_te,categorical_cols)
        tr_e=target_map(X_tr,y_tr,tr_e,cat_te_cols)
        va_e=target_map(X_tr,y_tr,va_e,cat_te_cols)
        te_e=target_map(X_tr,y_tr,te_e,cat_te_cols)
        add_cols=[c for c in tr_e.columns if c.endswith("__FREQ") or c.endswith("__TE")]
        tr_num=pd.concat([tr_num,tr_e[add_cols]],axis=1)
        va_num=pd.concat([va_num,va_e[add_cols]],axis=1)
        te_num=pd.concat([te_num,te_e[add_cols]],axis=1)
    imp=SimpleImputer(strategy="median",add_indicator=True)
    tr_num=imp.fit_transform(tr_num)
    va_num=imp.transform(va_num)
    te_num=imp.transform(te_num)
    for name,factory in models.items():
        model=factory()
        model.fit(tr_num,y_tr)
        pv=model.predict_proba(va_num)[:,1]
        pt=model.predict_proba(te_num)[:,1]
        oof[name][va_idx]=pv
        test_pred[name]+=pt/N_SPLITS
        fold_scores[name].append(roc_auc_score(y_va,pv))
    print(fold,{k:round(fold_scores[k][-1],6) for k in models})
model_scores={k:roc_auc_score(y,v) for k,v in oof.items()}
print(pd.Series(model_scores).sort_values(ascending=False))

1 {'lightgbm': np.float64(0.754745), 'xgboost': np.float64(0.756121), 'hgb': np.float64(0.754421)}
2 {'lightgbm': np.float64(0.759618), 'xgboost': np.float64(0.761551), 'hgb': np.float64(0.758277)}
3 {'lightgbm': np.float64(0.764429), 'xgboost': np.float64(0.765511), 'hgb': np.float64(0.761818)}
4 {'lightgbm': np.float64(0.753064), 'xgboost': np.float64(0.755051), 'hgb': np.float64(0.752829)}
5 {'lightgbm': np.float64(0.760851), 'xgboost': np.float64(0.760582), 'hgb': np.float64(0.754698)}
xgboost     0.759730
lightgbm    0.758495
hgb         0.756337
dtype: float64


In [10]:
names=list(oof)
best_score=max(model_scores.values())
best_weights={max(model_scores,key=model_scores.get):1.0}
if len(names)==2:
    for w in np.arange(0,1.001,0.025):
        p=w*oof[names[0]]+(1-w)*oof[names[1]]
        s=roc_auc_score(y,p)
        if s>best_score:
            best_score=s
            best_weights={names[0]:float(w),names[1]:float(1-w)}
elif len(names)>=3:
    grid=np.arange(0,1.001,0.05)
    for w0 in grid:
        for w1 in grid:
            if w0+w1>1: continue
            w2=1-w0-w1
            p=w0*oof[names[0]]+w1*oof[names[1]]+w2*oof[names[2]]
            s=roc_auc_score(y,p)
            if s>best_score:
                best_score=s
                best_weights={names[0]:float(w0),names[1]:float(w1),names[2]:float(w2)}
print(best_weights)
print(best_score)

{'lightgbm': 0.25, 'xgboost': 0.5, 'hgb': 0.25}
0.760820192970926


In [11]:
final_pred=np.zeros(len(X_test))
for name,w in best_weights.items():
    final_pred+=w*test_pred[name]
final_pred=np.clip(final_pred,0,1)
submission=sample_sub.copy()
submission["TARGET"]=final_pred
assert submission["SK_ID_CURR"].equals(test_raw["SK_ID_CURR"])
output_dir=current_dir/"output"
output_dir.mkdir(exist_ok=True)
output_file=output_dir/"submission_improved_v1_2.csv"
submission.to_csv(output_file,index=False)
print(output_file)
print(submission["TARGET"].describe())

/content/drive/MyDrive/GCI_Global/Competition/output/submission_improved_v1_2.csv
count    61500.000000
mean         0.093209
std          0.093127
min          0.002382
25%          0.030716
50%          0.060714
75%          0.121075
max          0.769161
Name: TARGET, dtype: float64


In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


In [13]:
def add_v14_features(df):
    df = df.copy()
    eps = 1e-6
    ext = [c for c in ["EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3"] if c in df.columns]
    if ext:
        x = df[ext]
        df["EXT_SOURCE_WEIGHTED"] = x.mean(axis=1, skipna=True)
        df["EXT_SOURCE_MIN"] = x.min(axis=1, skipna=True)
        df["EXT_SOURCE_MAX"] = x.max(axis=1, skipna=True)
        df["EXT_SOURCE_RANGE"] = df["EXT_SOURCE_MAX"] - df["EXT_SOURCE_MIN"]
        df["EXT_SOURCE_MEDIAN"] = x.median(axis=1, skipna=True)
        df["EXT_SOURCE_MISSING"] = x.isna().sum(axis=1)
        if "EXT_SOURCE_1" in df and "EXT_SOURCE_2" in df:
            df["EXT12_PRODUCT"] = df["EXT_SOURCE_1"] * df["EXT_SOURCE_2"]
            df["EXT12_DIFF"] = df["EXT_SOURCE_1"] - df["EXT_SOURCE_2"]
        if "EXT_SOURCE_1" in df and "EXT_SOURCE_3" in df:
            df["EXT13_PRODUCT"] = df["EXT_SOURCE_1"] * df["EXT_SOURCE_3"]
            df["EXT13_DIFF"] = df["EXT_SOURCE_1"] - df["EXT_SOURCE_3"]
        if "EXT_SOURCE_2" in df and "EXT_SOURCE_3" in df:
            df["EXT23_PRODUCT"] = df["EXT_SOURCE_2"] * df["EXT_SOURCE_3"]
            df["EXT23_DIFF"] = df["EXT_SOURCE_2"] - df["EXT_SOURCE_3"]
        for c in ext:
            df[c + "_SQ"] = df[c] ** 2
    if "AMT_CREDIT" in df and "AMT_INCOME_TOTAL" in df:
        df["CREDIT_INCOME_V14"] = df["AMT_CREDIT"] / (df["AMT_INCOME_TOTAL"].abs() + eps)
    if "AMT_ANNUITY" in df and "AMT_INCOME_TOTAL" in df:
        df["ANNUITY_INCOME_V14"] = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"].abs() + eps)
    if "AMT_CREDIT" in df and "AMT_ANNUITY" in df:
        df["CREDIT_ANNUITY_V14"] = df["AMT_CREDIT"] / (df["AMT_ANNUITY"].abs() + eps)
    if "AMT_GOODS_PRICE" in df and "AMT_CREDIT" in df:
        df["GOODS_CREDIT_V14"] = df["AMT_GOODS_PRICE"] / (df["AMT_CREDIT"].abs() + eps)
    if "DAYS_BIRTH" in df:
        df["AGE_YEARS_V14"] = -df["DAYS_BIRTH"] / 365.25
    if "DAYS_EMPLOYED" in df:
        df["EMPLOYED_ANOMALY_V14"] = (df["DAYS_EMPLOYED"] == 365243).astype("int8")
        emp = df["DAYS_EMPLOYED"].where(df["DAYS_EMPLOYED"] != 365243)
        df["EMPLOYED_YEARS_V14"] = -emp / 365.25
        if "DAYS_BIRTH" in df:
            df["EMPLOYED_AGE_RATIO_V14"] = (-emp) / (-df["DAYS_BIRTH"] + eps)
    if "CNT_FAM_MEMBERS" in df and "AMT_INCOME_TOTAL" in df:
        df["INCOME_PER_FAMILY_V14"] = df["AMT_INCOME_TOTAL"] / (df["CNT_FAM_MEMBERS"].clip(lower=1) + eps)
    miss = df.isna().sum(axis=1)
    df["MISSING_COUNT_V14"] = miss.astype("int16")
    df["MISSING_RATIO_V14"] = miss / max(1, df.shape[1])
    return df


In [14]:
train_v14 = add_v14_features(train_raw)
test_v14 = add_v14_features(test_raw)
X_v14 = train_v14.drop(columns=["TARGET"], errors="ignore")
y_v14 = train_v14["TARGET"].astype("int8")
X_test_v14 = test_v14.copy()


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

Xtr, Xva, ytr, yva = train_test_split(
    X_v14, y_v14, test_size=0.20, stratify=y_v14, random_state=42
)

cat_cols = Xtr.select_dtypes(include=["object","category"]).columns.tolist()
num_cols = [c for c in Xtr.columns if c not in cat_cols]

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=True), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=10))
    ]), cat_cols)
], remainder="drop")

lr_v14 = Pipeline([
    ("pre", pre),
    ("model", LogisticRegression(C=0.08, max_iter=500, solver="liblinear", class_weight="balanced"))
])

lr_v14.fit(Xtr, ytr)
p_lr_v14 = lr_v14.predict_proba(Xva)[:,1]
auc_lr_v14 = roc_auc_score(yva, p_lr_v14)
print("V14 Logistic AUC:", auc_lr_v14)


V14 Logistic AUC: 0.6668579180341955


In [16]:
blend_candidates_v14 = {
    "logistic": p_lr_v14
}
try:
    from lightgbm import LGBMClassifier
    lgb_v14 = LGBMClassifier(
        n_estimators=900,
        learning_rate=0.025,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.15,
        reg_lambda=1.5,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )
    lgb_v14.fit(Xtr, ytr, eval_set=[(Xva,yva)], callbacks=[])
    blend_candidates_v14["lightgbm"] = lgb_v14.predict_proba(Xva)[:,1]
    print("V14 LightGBM AUC:", roc_auc_score(yva, blend_candidates_v14["lightgbm"]))
except Exception as e:
    print("LightGBM unavailable:", str(e)[:160])


LightGBM unavailable: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: NAME_CONTRACT_TYPE: object, CODE_GENDER: object, NAME_INCOME_TYPE: object, NAME_EDUCATI


In [17]:
try:
    from xgboost import XGBClassifier
    xgb_v14 = XGBClassifier(
        n_estimators=700,
        max_depth=5,
        learning_rate=0.035,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=8,
        reg_alpha=0.05,
        reg_lambda=2.0,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )
    xgb_v14.fit(Xtr, ytr, eval_set=[(Xva,yva)], verbose=False)
    blend_candidates_v14["xgboost"] = xgb_v14.predict_proba(Xva)[:,1]
    print("V14 XGBoost AUC:", roc_auc_score(yva, blend_candidates_v14["xgboost"]))
except Exception as e:
    print("XGBoost unavailable:", str(e)[:160])


XGBoost unavailable: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must b


In [18]:
names = list(blend_candidates_v14)
best_auc_v14 = -1
best_name_v14 = None
best_weight_v14 = None

for name, pred in blend_candidates_v14.items():
    auc = roc_auc_score(yva, pred)
    if auc > best_auc_v14:
        best_auc_v14, best_name_v14, best_weight_v14 = auc, name, {name:1.0}

if "lightgbm" in blend_candidates_v14 and "xgboost" in blend_candidates_v14:
    for w in np.arange(0.2, 0.81, 0.1):
        pred = w*blend_candidates_v14["lightgbm"] + (1-w)*blend_candidates_v14["xgboost"]
        auc = roc_auc_score(yva, pred)
        if auc > best_auc_v14:
            best_auc_v14, best_name_v14, best_weight_v14 = auc, "lightgbm_xgboost", {"lightgbm":float(w),"xgboost":float(1-w)}

if "lightgbm" in blend_candidates_v14 and "logistic" in blend_candidates_v14:
    for w in np.arange(0.5, 0.91, 0.1):
        pred = w*blend_candidates_v14["lightgbm"] + (1-w)*blend_candidates_v14["logistic"]
        auc = roc_auc_score(yva, pred)
        if auc > best_auc_v14:
            best_auc_v14, best_name_v14, best_weight_v14 = auc, "lightgbm_logistic", {"lightgbm":float(w),"logistic":float(1-w)}

print("V14 best validation AUC:", best_auc_v14)
print("V14 best blend:", best_name_v14)
print("V14 weights:", best_weight_v14)


V14 best validation AUC: 0.6668579180341955
V14 best blend: logistic
V14 weights: {'logistic': 1.0}


In [21]:
final_pred=np.zeros(len(X_test))
for name,w in best_weights.items():
    final_pred+=w*test_pred[name]
final_pred=np.clip(final_pred,0,1)
submission=sample_sub.copy()
submission["TARGET"]=final_pred
assert submission["SK_ID_CURR"].equals(test_raw["SK_ID_CURR"])
output_dir=current_dir/"output"
output_dir.mkdir(exist_ok=True)
output_file=output_dir/"submission_improved_v1_4.csv"
submission.to_csv(output_file,index=False)
print(output_file)
print(submission["TARGET"].describe())


/content/drive/MyDrive/GCI_Global/Competition/output/submission_improved_v1_4.csv
count    61500.000000
mean         0.093209
std          0.093127
min          0.002382
25%          0.030716
50%          0.060714
75%          0.121075
max          0.769161
Name: TARGET, dtype: float64


In [20]:
import zipfile

# Define the file names
notebook_filename = "tutorial_improved_v1_4.ipynb" # Enter the name of this notebook
zip_filename = "tutorialv1.4.zip"

# Define the full paths using current_dir
notebook_path = current_dir / notebook_filename
zip_output_path = current_dir / zip_filename

# Check if the notebook file exists and create the ZIP file
if notebook_path.exists():
    with zipfile.ZipFile(zip_output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add the notebook to the ZIP file (arcname avoids including the full folder structure)
        zipf.write(notebook_path, arcname=notebook_filename)
    print(f"Successfully created: {zip_output_path}")
else:
    print(f"Error: Could not find '{notebook_filename}' in {current_dir}.")
    print("Please check if the notebook name and the current directory are correct.")

Successfully created: /content/drive/MyDrive/GCI_Global/Competition/tutorialv1.4.zip
